**Alunos:**. 
Eduardo Barbosa. 
Lucas Antonio Cunha Rodrigues da Silva. 
Luiz Girotto. 
Marcos Vinícius Beregula

**Curso:** Ciência de Dados e IA – 3º ano  
**Disciplina:** Aprendizado não supervisionado  
**Professor:** Gustavo Naozuka  
**Instituição:** Universidade Estadual de Londrina (UEL)  
**Data:** 24/06/2026  

# Imports
Definir as explicações

In [ ]:
from datasets import load_dataset

from itertools import islice
import json
import re
import unicodedata
from random import randint


import numpy as np

# Scikit Learning
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score

# Plotagem de gráficos
import matplotlib.pyplot as plt



# Dataset Wikipedia Structured
*O conjunto de dados contém todos os artigos das edições em inglês e francês da Wikipédia, pré-analisados ​​e apresentados como dados semiestruturados com um esquema consistente.  
O conjunto de dados é fornecido no formato Parquet, otimizado para consultas analíticas de alto desempenho e armazenamento eficiente. Esta versão utiliza um esquema unificado e fixo em todos os arquivos, tornando-a compatível com DuckDB, pandas, Polars e Apache Spark. Usamos "apenas" 1 milhao de linhas para obter aproximadamente 10 mil documentos com seccoes que superam as 1500 palavras*. 

In [ ]:
{
  "name": "Josephine Baker",
  "identifier": 255083,
  "url": "https://en.wikipedia.org/wiki/Josephine_Baker",
  "description": "American-born French dancer...",
  "abstract": "Freda Josephine Baker...",
  "main_entity": {
    "identifier": "Q151972",
    "url": "https://www.wikidata.org/entity/Q151972"
  },
  "version": {
    "identifier": 123456789,
    "editor": {
      "identifier": 123,
      "name": "Example editor"
    },
    "scores": {}
  },
  "image": {
    "content_url": "https://upload.wikimedia.org/...",
    "width": 250,
    "height": 350
  },
  "infoboxes": "[{...}]",
  "sections": "[{...}]",
  "tables": "[{...}]",
  "references": [
    {
      "identifier": "...",
      "metadata": "{...}"
    }
  ]
}

# 1.Preparação data-set


## 1.1 Funções auxiliares para limpeza dos textos

In [ ]:
#BLOCO FUNCIONAL - DECLARACAO DE FUNCOES - TRATAMENTO DOS DADOS TEXTUAIS:

def extrair_texto(doc):
    #extrair das secoes e concatenar em um unico texto

    try:
#devido a natureza semiestruturada, precisamos tratar excecoes para evitar erros
        texto = []
        secoes_dict = {}
        secoes = json.loads(doc["sections"])

        for secao in secoes:

            nome_secao = secao.get("name", "SEM_NOME")

            paragrafos_secao = []

            for parte in secao.get("has_parts", []):

                valor = parte.get("value")

                if (
                    parte.get("type") == "paragraph"
                    and isinstance(valor, str)
                ):

                    paragrafos_secao.append(valor)
                    texto.append(valor)

            if paragrafos_secao:

                secoes_dict[nome_secao] = " ".join(
                    paragrafos_secao
                )

        return {
            "texto": " ".join(texto),
            "secoes": secoes_dict
        }

    except Exception:

        return None
    
#funcao que captura o numero de palavras de cada documento
def tamanho_documento(doc):

    try:
        secoes = json.loads(doc["sections"])
        palavras = 0
        for secao in secoes:
            if "has_parts" not in secao:
                continue
            for parte in secao["has_parts"]:
                if parte.get("type") == "paragraph":
                    palavras += len(
                        parte["value"].split()
                    )
        return palavras
    except:
        return None

#funcao de limpeza de texto
def limpar_texto(texto):
    #MINUSCULAS
    texto = texto.lower()
    #NORMALIZACAO - SEPARA CAFÉ EM CAFE + ´ 
    texto = unicodedata.normalize("NFKD", texto)
    #USANDO ASCII COMO PADRAO  - REMOVE ACENTOS
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    #SUBSTITUINDO - REGEX >>> NAO ACEITA COISAS QUE NAO SAO LETRAS NUMEROS OU ESPACOS
    # NO LUGAR COLOCA ESPACO
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    #SUBSTITUINDO ESPACOS MAIORES QUE 1! POR UM ESPAÇO
    texto = re.sub(r"\s+", " ", texto)
    #STRIP REMOVE ESPACOS NO COMECO E FIM
    return texto.strip()

## 1.2 Filtragem dos documentos

In [ ]:
#carregando o dataset, ATENCAO NAO RODAR CASO FOR LER DO JSON, VAI DEMORAR MUITO
#PULAR ESTE BLOCO
ds = load_dataset(
    "wikimedia/structured-wikipedia",
    "enwiki_namespace_0",
    streaming=True
)
#1 MILHAO DE INSTANCIAS FORNECERAM 18 MIL DOCUMENTOS, RODEI ISSO DUAS VEZES PRA
#CHEGARMOS A 35 MIL DOCUMENTOS

amostra = list(islice(ds["train"],1000000))

#APENAS PRA CONFERENCIA DO PROCESSO INICIAL
#texto = extrair_texto(amostra[0])
#print("Palavras:", len(texto["texto"].split()))

tamanhos = []

amostra_completa = []

for doc in amostra:

    resultado = extrair_texto(doc)
    if resultado:
        texto = resultado["texto"]
        secoes = resultado["secoes"]
        n_palavras = len(texto.split())
        if n_palavras > 1500:

            tamanhos.append(n_palavras)
            documento = {
                "id": doc["identifier"],
                "titulo": doc["name"],
                "url": doc["url"],
                "texto": texto,
                "secoes": secoes,
                "n_palavras": n_palavras
            }

            amostra_completa.append(documento)

p25, p50, p75, p90, p95, p99 = np.percentile(
    tamanhos,
    [25, 50, 75, 90, 95, 99]
)

print(f"P25: {p25:.0f}")
print(f"Mediana: {p50:.0f}")
print(f"P75: {p75:.0f}")
print(f"P90: {p90:.0f}")
print(f"P95: {p95:.0f}")
print(f"P99: {p99:.0f}")
print(f"Média: {np.mean(tamanhos):.0f}")
print(f"Máximo: {np.max(tamanhos)}")
print(len(tamanhos))


## 1.3 Carregando Json para memória

In [ ]:
##PERSISTINDO OS DADOS EM UM JSON PARA EVITAR STREAMMING CONSTANTE
#NAO PRECISA RODAR CASO PEGUE O ARQUIVO JSON JA CRIADO APENAS A PARTE FINAL DESCOMENTADA
#with open(
    #"corpus_wikipedia_longos.json",
    #"w",
    #encoding="utf-8"
#) as f:

    #json.dump(
       # amostra_completa,
        #f,
        #ensure_ascii=False,
        #indent=2
    #)

#POSTERIORMENTE QUANDO FORMOS USAR O CONTEUDO TEXTUAL
with open(
    "data/corpus_wikipedia_longos.json",
    "r",
encoding="utf-8"
) as f:

    amostra_completa = json.load(f)

#AQUI VOLTAMOS A TER OS DADOS EM MEMORIA NA VARIAVEL amostra_completa

# 2. Vetorização

## 2.1 Vocabulário
Tratando as palavras

In [ ]:
#dimensionando o vocabulario, inferindo a dimensionalidade dos dados

# MEDINDO O VOCABULARIO PRA VER COMO ESTAMOS COM RELACAO A MAX FEATURES
# entra como parametro para o vetorizador (max.features)
vocabulario = set()

for doc in amostra_completa:
    vocabulario.update(doc["texto"].split())

print(f"Vocabulario antes do tratamento: {len(vocabulario)}")

#tratamento do dataset, ajuda na reducao de dimensionalidade

#TRATAMENTO DE TEXTO, CONFORME FUNCAO DE LIMPEZA (MMINUSCULO, ACENTOS E ESPECIAIS)


for k in amostra_completa:
    k["texto"] = limpar_texto(k["texto"])

#VERIFICANDO VOCABULARIO APOS TRATAMENTO

vocabulario = set()
for doc in amostra_completa:
    vocabulario.update(doc["texto"].split())

print(f"Vocabulario apos o tratamento: {len(vocabulario)}")

##como o JSON PERSISTIDO ja esta tratado voce vera valores iguais
#tinhamos 600k palavras reduzidas para 300k


## 2.2 Vetorização 
Estudar sobre implicação do ngram_range

In [ ]:
#vetorizaçao TF-IDF
textos = [doc["texto"] for doc in amostra_completa]
#CAPTURA LISTA DE TEXTOS

#CAPTURA LISTA DE IDS
ids = [doc["id"] for doc in amostra_completa]

#DECLARACAO DO VETORIZADOR
#ATENCAO SCIKIT-LEARN JA USA A NORMALIZACAO L2 COMO PADRAO (DEFAULT)!
vetorizador = TfidfVectorizer(
    stop_words="english",
    #max features LIMITA dimensionalidade excessiva
    max_features=10000,
    min_df=2, #ignorar palavras que aparecem em menos de 3 documentos
    max_df=0.9, #ignorar palavras que aparecem em mais de 90% dos documentos
    #adicionamos os bigramas e trigramas para capturar um pouco de contexto
    #desejamos realizar testes com e sem esse parametro tao importante
    ngram_range=(1, 3) #bigramas e trigramas
)

# Aqui tem que ver se o número de features é a len do vocabulario e a implicação da ngram_range

#APLICAÇÃO DO VETORIZADOR
X = vetorizador.fit_transform(textos)
print(X.shape)  # (24, ≤3000)

#VERIFICANDO OS n-GRAMAS GERADOS
recursos = vetorizador.get_feature_names_out()

trigrams_gerados = [termo for termo in recursos if len(termo.split()) == 3]

print(f"Total de trigrams que entraram no vocabulário: {len(trigrams_gerados)}")
print("Exemplos de trigrams encontrados:")
print(trigrams_gerados[:30]) # 30 primeiros

## 2.3 Redução de dimensionalidade
Verificar a utilização do SVD

In [ ]:
#ESMAGAMENTO DE DADOS - reduzindo a dimensionalidade
#SINGULAR VALUE DECOMPOSITION - REDUZIR A DIMENSIONALIDADE
#IDEIA MATEMATICA: QUALQUER MATRIZ PODE SER DECOMPOSTA EM 3 OUTRAS MATRIZES
#U * SIGMA * V^T
#U E V SAO ORTOGONAIS E SIGMA EH DIAGONAL
#SIGMA EH UMA MATRIZ QUE CONTÉM OS VALORES SINGULARES DA MATRIZ X
#OS VALORES SINGULARES SAO OS VALORES DAS COLUNAS DE U E V
#A MATRIZ SIGMA EH UMA MATRIZ DIAGONAL QUE CONTÉM OS VALORES SINGULARES

#Em resumo o melhor subespeço que para representar a matriz de documentos

# CORRECAO: n=50 maximiza sil_cosine no sweep (validacao_estrutura_clusters.ipynb)
SVD_COMPONENTS = 50

svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
X_svd = svd.fit_transform(X)

print(X_svd.shape)

# espaco reduzido normalizado para o fskmeans (nao alterar X esparso do TF-IDF)
X_cluster = normalize(X_svd, norm="l2", axis=1)

variancia_retida = np.sum(svd.explained_variance_ratio_) * 100
print(f"O SVD conseguiu reter {variancia_retida:.2f}% da informação original dos textos.")

termos = vetorizador.get_feature_names_out()

#visualizar os topicos latentes (3 PRIMEIROS), podemos ver mais tambem
for i, topico in enumerate(svd.components_[:3]):
    # Pega os índices dos 10 termos com maior peso no tópico
    top_termos_idx = topico.argsort()[::-1][:10]
    top_termos = [termos[idx] for idx in top_termos_idx]
    print(f"\nTópico Latente {i+1}:")
    print(", ".join(top_termos))


# 3. Algoritmo

## 3.1 Inicialização dos centróides
Utilização do k-means ++

In [ ]:
#ETAPA ALGORITMO FSK-MEANS (pipeline corrigido)

#INICIALIZACAO ++ (k-means++ no espaco SVD, nao no TF-IDF esparso)

FSK_M = 1.8
K_CLUSTERS = 12

k = K_CLUSTERS
eps = 1e-7
max_iter = 1000
m = FSK_M

n_docs = X_cluster.shape[0]
rng = np.random.default_rng(42)

centroides = X_cluster[rng.integers(n_docs)].reshape(1, -1)

for _ in range(k - 1):
    sims = X_cluster @ centroides.T
    distancias = 1 - np.max(sims, axis=1)
    distancias = np.clip(distancias, 1e-12, None)
    prob = distancias / distancias.sum()
    novo_idx = rng.choice(n_docs, p=prob)
    centroides = np.vstack([centroides, X_cluster[novo_idx]])

print(f"Centroides inicializados: {centroides.shape}")

## 3.2 Funções auxiliares do Algoritmo FSKmeans

In [ ]:
#BLOCO FUNCIONAL - DECLARACAO DE FUNCOES - ALGORITMO FSK-MEANS

#IMPLEMENTACAO MANUAL SEM USO DE BIBLIOTECAS!!! PODE SER PESADO 

#def calcular_distancias(X_svd, centroides):
    #distancias = []
    #for i in range(len(X_svd)):
        #dist = []
        #for j in range(len(centroides)):
            #distancia = np.linalg.norm(X_svd[i] - centroides[j])
            #dist.append(distancia)
        #distancias.append(dist)
    #return distancias
    
#def atualizar_centroides(X, pertencimentos, m):
    #FRACA EM PERFORMANCE, MAS FUNCIONA, IMPLEMENTACAO NA RAÇA
    #centroides = []
    #linhas, k = pertencimentos.shape
    #dimensao = X.shape[1]
    #for i in range(k):
    #COLUNAS REPRESENTAM K CENTROIDES, APENAS PRA NAO PRECISAR PASSAR O PARAMETRO K
        #numerador = np.zeros(dimensao)
        #denominador = 0
        #for j in range(linhas):
            #LINHAS REPRESENTAM NUMERO DEDOCUMENTOS
            #parcial = (pertencimentos[j][i] ** m)
            #numerador += parcial * X[j]
            #denominador += parcial
        #centroides.append(numerador / denominador)
    #NORMALIZACAO L2
    #centroides = np.array(centroides)
    #centroides = centroides / np.linalg.norm(centroides, axis=1, keepdims=True)
    #return np.array(centroides)


#CALCULO DOS PERTENCIMENTOS MI'S - funcao sem algebra vetorial
#def calcular_pertencimentos(distancias, m):
    #pertencimentos = []
    #eps = 1e-10
    #for i in range(len(distancias)):
        #linha = []
        #n_clusters = len(distancias[i])
        #for j in range(n_clusters):
            #soma = 0
            #for l in range(n_clusters):
                #evitando divisão por zero
#MELHORIA SE NAO HA DISTANCIA PERTENCIMENTO EH TOTAL (1) RETORNAR 1 NESTE MOMENTO
                #dij = max(distancias[i][j], eps)
                #dil = max(distancias[i][l], eps)
                #parcial = ((dij / dil)) ** (2 / (m - 1))
                #soma += parcial
            #pertencimento = 1 / soma
            #linha.append(pertencimento)
        #pertencimentos.append(linha)
    #return pertencimentos

#IMPLEMENTACAO COM USO DE ALGEBRA VETORIAL E BROADCASTING:

def calcular_distancias_algebrica(X, centroides):
    # Distancia angular para vetores L2-normalizados: d = 1 - cos(x, c)
    X = np.asarray(X)
    centroides = np.asarray(centroides)
    sims = X @ centroides.T
    distancias = 1.0 - sims
    return np.clip(distancias, 1e-12, 2.0)

def calcular_pertencimentos_algebrica(distancias, m):
    eps = 1e-10
    # evitando divisão por zero
    distancias_max = np.maximum(distancias, eps)
    expoente = 2 / (m - 1)
    # broadcasting: (n,k,1) / (n,1,k)
    ratio = (distancias_max[:, :, None] / distancias_max[:, None, :]) ** expoente
    soma = np.sum(ratio, axis=2)
    pertencimentos = 1 / soma
    return pertencimentos

def atualizar_centroides_algebrica(X, pertencimentos, m):
    X = np.asarray(X)
    PTimesM = pertencimentos ** m  #(n_docs, k)
    numerador = PTimesM.T @ X  #(k, n_docs) vezes (n_docs, d) = (k, d)
    denominador = np.sum(PTimesM, axis=0).reshape(-1, 1)  #(k, 1)
    denominador = np.maximum(denominador, 1e-7)

    centroides = numerador / denominador

    # projeção de volta na esfera - normalização L2
    centroides = centroides / np.linalg.norm(centroides, axis=1, keepdims=True)
    return centroides

# Hiperesfera
def min_cosine_centroides(cent):
    c = normalize(np.asarray(cent), norm="l2", axis=1)
    sim = c @ c.T
    np.fill_diagonal(sim, -1.0)
    return sim.max()

def entropia_fuzzy(P):
    P = np.clip(P, 1e-12, 1.0)
    return (-P * np.log(P)).sum(axis=1).mean()
    

## 3.3 Clusterização

In [ ]:
#BLOCO DE OBTENCAO DOS CENTROIDES FINAIS E PERTENCIMENTOS

convergencia = False
iteracao = 0
historico = []
while not convergencia and iteracao < max_iter:
    distancias = calcular_distancias_algebrica(X_cluster, centroides)
    pertencimentos = calcular_pertencimentos_algebrica(distancias, m)
    centroides_novos = atualizar_centroides_algebrica(X_cluster, pertencimentos, m)
    delta = np.linalg.norm(centroides_novos - centroides)

    min_cos = min_cosine_centroides(centroides_novos)
    ent = entropia_fuzzy(pertencimentos)
    ent_max = np.log(k)

    if min_cos > 0.95 or ent > 0.95 * ent_max:
        print(f"ALERTA colapso iter {iteracao}: min_cos={min_cos:.4f}, ent={ent:.3f}/{ent_max:.3f}")

    if delta < eps:
        convergencia = True

    jayM = np.sum((pertencimentos ** m) * (distancias ** 2))
    historico.append(jayM)
    print(
        f"Iter {iteracao}: delta={delta:.8f} | J={jayM:.4f} | "
        f"min_cos_cent={min_cos:.4f} | ent={ent:.3f}"
    )

    centroides = centroides_novos
    iteracao += 1

plt.figure(figsize=(6, 3))
plt.plot(historico)
plt.title("FSK-means: funcao objetivo J")
plt.xlabel("iteracao")
plt.tight_layout()
plt.show()

## 3.4 Data frame clusters

In [ ]:
#BLOCO DE OBTENCAO DOS CLUSTERS, SALVANDO EM DICIONARIOS

import pandas as pd

rotulos = np.argmax(pertencimentos, axis=1)

df_clusters = pd.DataFrame({
    "id": ids,
    "cluster_dominante": rotulos
})

for c in range(pertencimentos.shape[1]):
    df_clusters[f"cluster_{c}"] = pertencimentos[:, c]

df_clusters


In [ ]:
# Filtra apenas as colunas que começam com 'cluster_' ignorando a 'cluster_dominante'
colunas_clusters = [col for col in df_clusters.columns if col.startswith('cluster_') and col != 'cluster_dominante']

# Soma os valores dessas colunas no eixo 1 (linhas)
df_clusters['soma_clusters'] = df_clusters[colunas_clusters].sum(axis=1)

df_clusters

## 3.5 Data frame centroides

In [ ]:

print(centroides)

print("\nDISTÂNCIAS ENTRE CENTROIDES:")
print(np.linalg.norm(centroides[:, None, :] - centroides[None, :, :], axis=2))


# 4. Otimização de hiperparâmetros

## 4.1 Shilhoutte

## 4.2 Elbow

# X. Comparativo com baseline KMeans (REVISAR SE VAI FICAR)

In [ ]:
# Comparacao: KMeans baseline vs FSK-means (executar APOS o treino acima)
from sklearn.cluster import KMeans

km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_cluster)
rotulos_km = km.labels_
cent_km = normalize(km.cluster_centers_, norm="l2", axis=1)
min_cos_km = (cent_km @ cent_km.T)[np.triu_indices(k, 1)].min()

rotulos_fsk = np.argmax(pertencimentos, axis=1)
min_cos_fsk = min_cosine_centroides(centroides)
ent = entropia_fuzzy(pertencimentos)
ent_max = np.log(k)

print("=== METRICAS POS-CORRECAO ===")
print(f"SVD={SVD_COMPONENTS} | k={k} | m={m}")
print(f"KMeans  sil_cosine={silhouette_score(X_cluster, rotulos_km, metric='cosine'):.4f} | min_cos_cent={min_cos_km:.4f}")
print(f"FSK     sil_cosine={silhouette_score(X_cluster, rotulos_fsk, metric='cosine'):.4f} | min_cos_cent={min_cos_fsk:.4f}")
print(f"FSK entropia={ent:.3f}/{ent_max:.3f} ({ent/ent_max:.1%}) | std(mu)={pertencimentos.std(axis=1).mean():.4f}")
if min_cos_fsk > 0.95 or ent > 0.95 * ent_max:
    print("ALERTA: possivel colapso fuzzy")
else:
    print("FSK: pertencimentos nao uniformes (sem colapso)")

# 5. Métricas de avaliação

## 5.2 Davies bouldin

## 5.2 Harabasz

# 6. Apresentação de resultados

## 6.1 Gráfico de dispersão 3D
Redução de dimensionalidade (provável PCA) para 3 dimensões e plotagem do gráfico com o fuzzy.


## 6.2 Análise manual de textos do mesmo cluster

## 6.3 Nomear os clusters